# Company Canonicalization

This notebook canonicalizes company names using fuzzy matching, aliases, and entity resolution to map variations to canonical company identifiers.

## Architecture

**Input**: `workspace.silver.silver_jobs_current`  
**Output**: `workspace.semantic.sem_company_map`  
**Review Queue**: `workspace.silver.silver_semantic_review_queue`  
**Mode**: Incremental (process new batches only)

## Batch Processing

- Tracks processed batches in `metadata.company_canonicalization_batches`
- Automatically processes all unprocessed batches
- Idempotent: safe to re-run
- Confidence scoring at each stage

## Matching Strategy
1. **Exact Match**: Direct lookup in canonical company table
2. **Alias Match**: Check known aliases and abbreviations
3. **Fuzzy Match**: Levenshtein distance and token-based similarity
4. **Domain Match**: Match by company website/domain
5. **Review Queue**: Low-confidence matches flagged for review

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for all unprocessed)")

batch_id = dbutils.widgets.get("batch_id").strip()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, TimestampType
from pyspark.sql.window import Window
from datetime import datetime
import re

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
SEMANTIC_SCHEMA = f"{CATALOG}.semantic"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Configuration
CONFIG = {
    "input_table": f"{SILVER_SCHEMA}.silver_jobs_current",
    "output_table": f"{SEMANTIC_SCHEMA}.sem_company_map",
    "review_queue_table": f"{SILVER_SCHEMA}.silver_semantic_review_queue",
    "canonical_companies_table": f"{SEMANTIC_SCHEMA}.canonical_companies",
    "confidence_threshold": 0.75,
    "fuzzy_match_threshold": 0.85,
    "min_string_length": 3,
    "remove_suffixes": ["inc", "corp", "corporation", "ltd", "limited", "llc", "co", "company"]
}

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

In [0]:
# Utility functions for company name normalization

def normalize_company_name(name):
    """Normalize company name for matching."""
    if not name:
        return ""
    
    # Convert to lowercase
    name = name.lower().strip()
    
    # Remove common punctuation
    name = re.sub(r'[.,\-_]', ' ', name)
    
    # Remove multiple spaces
    name = re.sub(r'\s+', ' ', name)
    
    # Remove common suffixes
    for suffix in CONFIG["remove_suffixes"]:
        pattern = r'\b' + suffix + r'\b'
        name = re.sub(pattern, '', name)
    
    return name.strip()

def levenshtein_distance(s1, s2):
    """Calculate Levenshtein distance between two strings."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

def string_similarity(s1, s2):
    """Calculate similarity score between two strings (0-1)."""
    if not s1 or not s2:
        return 0.0
    
    distance = levenshtein_distance(s1, s2)
    max_len = max(len(s1), len(s2))
    
    if max_len == 0:
        return 1.0
    
    return 1.0 - (distance / max_len)



In [0]:
# Create metadata table to track company-canonicalized batches
metadata_table = f"{METADATA_SCHEMA}.company_canonicalization_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  companies_processed INT,
  canonical_count INT,
  review_queue_count INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING
)
USING DELTA
COMMENT 'Tracks which batches have been company-canonicalized'
""")

# Define metadata schema for recording batch processing
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("companies_processed", IntegerType(), True),
    StructField("canonical_count", IntegerType(), True),
    StructField("review_queue_count", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# ⚠️ UNCOMMENT TO RESET: Deletes all company mappings and metadata to reprocess everything
# Use this when dictionary/logic changes require full reprocessing

# print("⚠️  RESETTING: Deleting all company mappings and metadata...")
# spark.sql(f"DELETE FROM {CONFIG['output_table']}")
# spark.sql(f"DELETE FROM {metadata_table}")
# spark.sql(f"DELETE FROM {CONFIG['review_queue_table']} WHERE issue_type IN ('low_confidence', 'no_match')")
# print("✅ Reset complete. All batches will be reprocessed with updated matching logic.")

In [0]:
# Identify unprocessed batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
    print(f"Processing specific batch: {batch_id}")
else:
    # Find all batches in current table
    all_batches_df = spark.table(CONFIG["input_table"]).select(
        F.col("current_batch_id").alias("batch_id"),
        F.col("source_name")
    ).distinct()
    
    # Find already processed batches
    processed_batches_df = spark.table(metadata_table).select(
        F.col("batch_id"),
        F.col("source_name")
    ).distinct()
    
    # Get unprocessed batches
    unprocessed_df = all_batches_df.join(
        processed_batches_df,
        ["batch_id", "source_name"],
        "left_anti"
    )
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unprocessed_df.collect()]
    print(f"Found {len(batches_to_process)} unprocessed batch(es) to process")

In [0]:
# Create canonical companies table if it doesn't exist
canonical_companies_table = CONFIG["canonical_companies_table"]

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {canonical_companies_table} (
  company_id STRING,
  canonical_company_name STRING,
  domain STRING,
  industry STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Canonical company reference - auto-populated from unmatched companies'
""")

# Seed with initial companies if table is empty
if spark.table(canonical_companies_table).count() == 0:
    print("Seeding canonical companies with initial data...")
    sample_companies = [
        ("company_0001", "Apple Inc", "apple.com", "Technology"),
        ("company_0002", "Microsoft Corporation", "microsoft.com", "Technology"),
        ("company_0003", "Google LLC", "google.com", "Technology"),
        ("company_0004", "Amazon.com Inc", "amazon.com", "E-commerce"),
        ("company_0005", "Meta Platforms Inc", "meta.com", "Technology"),
    ]
    
    seed_df = spark.createDataFrame(
        sample_companies,
        ["company_id", "canonical_company_name", "domain", "industry"]
    ).withColumn("created_at", F.current_timestamp()) \
     .withColumn("updated_at", F.current_timestamp())
    
    seed_df.write.format("delta").mode("append").saveAsTable(canonical_companies_table)
    print(f"✓ Seeded {len(sample_companies)} companies")

# Load canonical companies from table
canonical_companies_df = spark.table(canonical_companies_table)
print(f"Loaded {canonical_companies_df.count()} canonical companies from {canonical_companies_table}")

# Load company aliases (for now, keep as hardcoded until we have an aliases table)
sample_aliases = [
    ("company_0001", "Apple", "common_name"),
    ("company_0001", "Apple Computer", "historical_name"),
    ("company_0002", "Microsoft", "common_name"),
    ("company_0002", "MSFT", "ticker"),
    ("company_0003", "Google", "common_name"),
    ("company_0003", "Alphabet", "parent_company"),
    ("company_0004", "Amazon", "common_name"),
    ("company_0005", "Meta", "common_name"),
    ("company_0005", "Facebook", "historical_name"),
    ("company_0005", "FB", "ticker"),
]

company_aliases_df = spark.createDataFrame(
    sample_aliases,
    ["company_id", "alias", "alias_type"]
)


In [0]:
# Load companies from unprocessed batches only
if len(batches_to_process) == 0:
    print("\n✓ No unprocessed batches found. All companies are already canonicalized.")
    dbutils.notebook.exit('{"status": "no_batches", "message": "All batches already processed"}')

print(f"\nLoading companies from {len(batches_to_process)} unprocessed batch(es)...")

# Get batch IDs to process
batch_ids_to_process = [batch_id for batch_id, _ in batches_to_process]

# Load companies only from unprocessed batches
jobs_df = spark.table(CONFIG["input_table"])

extracted_companies_df = jobs_df.filter(
    F.col("current_batch_id").isin(batch_ids_to_process)
).select(
    F.col("enterprise_job_id").alias("extraction_id"),
    F.col("company_name_norm").alias("extracted_company_name"),
    F.col("enterprise_job_id").alias("source_id"),
    F.col("current_batch_id").alias("batch_id"),
    F.col("source_name")
).filter(
    F.col("company_name_norm").isNotNull()
).distinct()

extracted_count = extracted_companies_df.count()
print(f"✓ Loaded {extracted_count} unique company names from {len(batches_to_process)} batch(es)")

if extracted_count == 0:
    print("\n⚠ No companies to process in these batches.")
    dbutils.notebook.exit('{"status": "no_companies", "message": "No companies found in unprocessed batches"}')

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Register normalization UDF
normalize_udf = udf(normalize_company_name, StringType())

# Normalize extracted company names (preserve batch_id and source_name)
extracted_normalized_df = extracted_companies_df.withColumn(
    "normalized_name",
    normalize_udf(F.col("extracted_company_name"))
).filter(
    F.length(F.col("normalized_name")) >= CONFIG["min_string_length"]
)

filtered_count = extracted_normalized_df.count()
print(f"After normalization and filtering: {filtered_count} companies")
print(f"Filtered out {extracted_count - filtered_count} companies (too short or invalid)")

display(extracted_normalized_df)

In [0]:
# Define normalization UDF (needed for this cell)
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

normalize_udf = udf(normalize_company_name, StringType())

# Build normalized canonical companies lookup
canonical_normalized_df = canonical_companies_df.withColumn(
    "normalized_name",
    normalize_udf(F.col("canonical_company_name"))
)

print("Canonical companies (normalized):")
display(canonical_normalized_df)

# Build aliases lookup
aliases_normalized_df = company_aliases_df.withColumn(
    "normalized_alias",
    normalize_udf(F.col("alias"))
)

print("\nCompany aliases (normalized):")
display(aliases_normalized_df)

In [0]:
# Exact match on normalized names
exact_matches_df = extracted_normalized_df.join(
    canonical_normalized_df.select("company_id", "canonical_company_name", "normalized_name"),
    extracted_normalized_df.normalized_name == canonical_normalized_df.normalized_name,
    "left"
).select(
    extracted_normalized_df.extraction_id,
    extracted_normalized_df.extracted_company_name,
    extracted_normalized_df.source_id,
    extracted_normalized_df.normalized_name,
    extracted_normalized_df.batch_id,
    extracted_normalized_df.source_name,
    F.col("company_id"),
    F.col("canonical_company_name"),
    F.lit("exact").alias("match_method"),
    F.lit(1.0).alias("confidence")
)

matched_df = exact_matches_df.filter(F.col("company_id").isNotNull())
unmatched_df = exact_matches_df.filter(F.col("company_id").isNull()).drop("company_id", "canonical_company_name", "match_method", "confidence")

matched_count = matched_df.count()
unmatched_count = unmatched_df.count()

print(f"Stage 1 - Exact Match Results:")
print(f"  Matched: {matched_count}")
print(f"  Unmatched: {unmatched_count}")

if matched_count > 0:
    print("\nExact matches:")
    display(matched_df)

In [0]:
# Match against aliases
alias_matches_df = unmatched_df.join(
    aliases_normalized_df.select("company_id", "alias", "normalized_alias", "alias_type"),
    unmatched_df.normalized_name == aliases_normalized_df.normalized_alias,
    "left"
).join(
    canonical_companies_df.select("company_id", "canonical_company_name"),
    "company_id",
    "left"
).select(
    unmatched_df.extraction_id,
    unmatched_df.extracted_company_name,
    unmatched_df.source_id,
    unmatched_df.normalized_name,
    unmatched_df.batch_id,
    unmatched_df.source_name,
    F.col("company_id"),
    F.col("canonical_company_name"),
    F.lit("alias").alias("match_method"),
    F.lit(0.95).alias("confidence")
)

alias_matched_df = alias_matches_df.filter(F.col("company_id").isNotNull())
alias_unmatched_df = alias_matches_df.filter(F.col("company_id").isNull()).drop("company_id", "canonical_company_name", "match_method", "confidence")

alias_matched_count = alias_matched_df.count()
alias_unmatched_count = alias_unmatched_df.count()

print(f"\nStage 2 - Alias Match Results:")
print(f"  Matched: {alias_matched_count}")
print(f"  Unmatched: {alias_unmatched_count}")

if alias_matched_count > 0:
    print("\nAlias matches:")
    display(alias_matched_df)

# Combine matched results
all_matched_df = matched_df.union(alias_matched_df)

In [0]:
from pyspark.sql.functions import udf, struct, max as spark_max
from pyspark.sql.types import FloatType, StructType, StructField

# Define fuzzy matching UDF
fuzzy_match_schema = StructType([
    StructField("company_id", StringType(), True),
    StructField("canonical_name", StringType(), True),
    StructField("similarity", FloatType(), True)
])

def fuzzy_match_company(normalized_name, canonical_list):
    """Find best fuzzy match from canonical companies."""
    if not normalized_name or not canonical_list:
        return (None, None, 0.0)
    
    best_match = None
    best_similarity = 0.0
    
    for company_id, canonical_name in canonical_list:
        sim = string_similarity(normalized_name, canonical_name)
        if sim > best_similarity:
            best_similarity = sim
            best_match = (company_id, canonical_name)
    
    if best_match and best_similarity >= CONFIG["fuzzy_match_threshold"]:
        return (best_match[0], best_match[1], best_similarity)
    
    return (None, None, best_similarity)

# Collect canonical companies for broadcast
canonical_list = [(row.company_id, row.normalized_name) 
                   for row in canonical_normalized_df.select("company_id", "normalized_name").collect()]

print(f"\nStage 3 - Fuzzy Matching against {len(canonical_list)} canonical companies...")
print(f"Fuzzy match threshold: {CONFIG['fuzzy_match_threshold']}")

# For demonstration, we'll use a simpler approach with cross join and filtering
# In production, consider using locality-sensitive hashing or other scalable methods

fuzzy_matches_df = alias_unmatched_df.crossJoin(
    canonical_normalized_df.select(
        F.col("company_id"),
        F.col("canonical_company_name"),
        F.col("normalized_name").alias("canonical_normalized")
    )
)

# Calculate similarity for each pair
similarity_udf = udf(string_similarity, FloatType())

fuzzy_scored_df = fuzzy_matches_df.withColumn(
    "similarity",
    similarity_udf(F.col("normalized_name"), F.col("canonical_normalized"))
).filter(
    F.col("similarity") >= CONFIG["fuzzy_match_threshold"]
)

# Keep best match per extracted company
window_spec = Window.partitionBy("extraction_id").orderBy(F.desc("similarity"))

fuzzy_best_matches_df = fuzzy_scored_df.withColumn(
    "rank",
    F.row_number().over(window_spec)
).filter(
    F.col("rank") == 1
).select(
    "extraction_id",
    "extracted_company_name",
    "source_id",
    "normalized_name",
    "batch_id",
    "source_name",
    "company_id",
    "canonical_company_name",
    F.lit("fuzzy").alias("match_method"),
    F.col("similarity").alias("confidence")
).drop("rank")

fuzzy_matched_count = fuzzy_best_matches_df.count()
fuzzy_unmatched_count = alias_unmatched_count - fuzzy_matched_count

print(f"  Matched: {fuzzy_matched_count}")
print(f"  Unmatched: {fuzzy_unmatched_count}")

if fuzzy_matched_count > 0:
    print("\nFuzzy matches:")
    display(fuzzy_best_matches_df.orderBy(F.desc("confidence")))

# Get truly unmatched records
fuzzy_unmatched_df = alias_unmatched_df.join(
    fuzzy_best_matches_df.select("extraction_id"),
    "extraction_id",
    "left_anti"
)

# Combine all matched results
all_matched_df = all_matched_df.union(fuzzy_best_matches_df)

In [0]:
# Split matches by confidence threshold
high_confidence_df = all_matched_df.filter(
    F.col("confidence") >= CONFIG["confidence_threshold"]
).withColumn(
    "processed_timestamp",
    F.current_timestamp()
)

low_confidence_df = all_matched_df.filter(
    F.col("confidence") < CONFIG["confidence_threshold"]
).withColumn(
    "review_reason",
    F.lit("low_confidence")
).withColumn(
    "review_status",
    F.lit("pending")
).withColumn(
    "processed_timestamp",
    F.current_timestamp()
)

# Add unmatched to review queue (preserve batch fields)
unmatched_review_df = fuzzy_unmatched_df.select(
    "extraction_id",
    "extracted_company_name",
    "source_id",
    "normalized_name",
    "batch_id",
    "source_name",
    F.lit(None).cast(StringType()).alias("company_id"),
    F.lit(None).cast(StringType()).alias("canonical_company_name"),
    F.lit("no_match").alias("match_method"),
    F.lit(0.0).alias("confidence"),
    F.lit("no_match").alias("review_reason"),
    F.lit("pending").alias("review_status"),
    F.current_timestamp().alias("processed_timestamp")
)

review_queue_df = low_confidence_df.union(unmatched_review_df)

high_conf_count = high_confidence_df.count()
low_conf_count = low_confidence_df.count()
unmatched_review_count = unmatched_review_df.count()

print(f"\nMatching Summary:")
print(f"  High confidence (>= {CONFIG['confidence_threshold']}): {high_conf_count}")
print(f"  Low confidence (< {CONFIG['confidence_threshold']}): {low_conf_count}")
print(f"  No match: {unmatched_review_count}")
print(f"  Total for review: {review_queue_df.count()}")
print(f"  Success rate: {high_conf_count / filtered_count * 100:.1f}%")

In [0]:
# Prepare data for sem_company_map schema
from pyspark.sql.functions import md5, concat, lit, when

output_df = high_confidence_df.select(
    md5(concat(F.col("extraction_id"), F.col("company_id"))).alias("company_map_id"),
    F.col("extraction_id").alias("enterprise_job_id"),
    F.col("company_id").alias("canonical_company_id"),
    F.col("canonical_company_name"),
    when(F.col("match_method") == "exact", "EXACT_MATCH")
     .when(F.col("match_method") == "alias", "ALIAS_MATCH")
     .when(F.col("match_method") == "fuzzy", "FUZZY")
     .otherwise("MANUAL").alias("normalization_method"),
    F.col("confidence").cast("decimal(5,4)").alias("normalization_confidence"),
    F.to_json(F.struct(
        F.col("extracted_company_name").alias("input"),
        F.col("normalized_name").alias("normalized"),
        F.col("match_method").alias("method")
    )).alias("explanation_json"),
    F.col("batch_id").alias("effective_batch_id")
)

# STEP 1: Add unmatched companies as new canonical entries
if unmatched_review_count > 0:
    print(f"\nℹ Adding {unmatched_review_count} unmatched companies as new canonical entries...")
    
    # Generate new company IDs
    from pyspark.sql.functions import monotonically_increasing_id, lpad
    
    new_canonical_df = fuzzy_unmatched_df.select(
        F.concat(
            F.lit("company_"),
            F.lpad(F.monotonically_increasing_id().cast("string"), 8, "0")
        ).alias("company_id"),
        F.col("extracted_company_name").alias("canonical_company_name"),
        F.lit(None).cast(StringType()).alias("domain"),
        F.lit("Unknown").alias("industry"),
        F.current_timestamp().alias("created_at"),
        F.current_timestamp().alias("updated_at")
    ).distinct()
    
    # Write new canonical companies
    new_canonical_df.write.format("delta").mode("append").saveAsTable(CONFIG["canonical_companies_table"])
    print(f"✓ Added {new_canonical_df.count()} new canonical companies to {CONFIG['canonical_companies_table']}")
    print("  ℹ These will be available for matching in the next run")

# STEP 2: Write high-confidence matches to sem_company_map
print(f"\nWriting {high_conf_count} company mappings to {CONFIG['output_table']}...")

# Use MERGE for idempotency
from delta.tables import DeltaTable

if spark.catalog.tableExists(CONFIG["output_table"]):
    target_table = DeltaTable.forName(spark, CONFIG["output_table"])
    target_table.alias("target").merge(
        output_df.alias("source"),
        "target.company_map_id = source.company_map_id"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print("✓ Company mappings merged successfully (idempotent)")
else:
    output_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["output_table"])
    print("✓ Company mappings written successfully")

# Write review queue
review_count = review_queue_df.count()
if review_count > 0:
    print(f"\nWriting {review_count} records to review queue at {CONFIG['review_queue_table']}...")
    
    review_df = review_queue_df.select(
        F.expr("uuid()").alias("review_id"),
        F.col("extraction_id").alias("enterprise_job_id"),
        F.col("review_reason").alias("issue_type"),
        F.concat(
            F.lit("Company: "),
            F.col("extracted_company_name"),
            F.lit(", Normalized: "),
            F.col("normalized_name"),
            F.lit(", Suggested: "),
            F.coalesce(F.col("canonical_company_name"), F.lit("none")),
            F.lit(", Batch: "),
            F.col("batch_id")
        ).alias("issue_detail"),
        F.coalesce(F.col("confidence"), F.lit(0.0)).cast("decimal(5,4)").alias("confidence"),
        F.current_timestamp().alias("queued_at"),
        F.col("review_status").alias("status")
    )
    
    review_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["review_queue_table"])
    
    print("✓ Review queue records written successfully")

# Write batch metadata for tracking
print(f"\nWriting batch metadata for {len(batches_to_process)} batch(es)...")

batch_metadata_records = []
for batch_id_val, source_name_val in batches_to_process:
    # Calculate per-batch metrics
    batch_high_conf = high_confidence_df.filter(F.col("batch_id") == batch_id_val).count()
    batch_review = review_queue_df.filter(F.col("batch_id") == batch_id_val).count() if "batch_id" in review_queue_df.columns else 0
    batch_total = extracted_companies_df.filter(F.col("batch_id") == batch_id_val).count()
    
    batch_metadata_records.append((
        batch_id_val,
        source_name_val,
        batch_total,
        batch_high_conf,
        batch_review,
        run_timestamp_py,
        run_id,
        "success"
    ))

metadata_df = spark.createDataFrame(batch_metadata_records, schema=metadata_schema)
metadata_df.write.format("delta").mode("append").saveAsTable(metadata_table)
print("✓ Batch metadata written successfully")

print("\n" + "="*60)
print("COMPANY CANONICALIZATION - EXECUTION SUMMARY")
print("="*60)
print(f"Batches processed: {len(batches_to_process)}")
print(f"Input companies: {filtered_count}")
print(f"High confidence matches: {high_conf_count} ({high_conf_count/filtered_count*100:.1f}%)")
print(f"Review queue: {review_queue_df.count()} ({review_queue_df.count()/filtered_count*100:.1f}%)")
print(f"\nMatch method breakdown:")
method_stats = high_confidence_df.groupBy("match_method").count().orderBy(F.desc("count"))
for row in method_stats.collect():
    print(f"  {row['match_method']}: {row['count']}")
print("="*60)

In [0]:
# Return summary as JSON for pipeline orchestration
import json

summary = {
    "status": "success",
    "run_id": run_id,
    "batches_processed": len(batches_to_process),
    "companies_processed": filtered_count,
    "canonical_count": high_conf_count,
    "review_queue_count": review_queue_df.count(),
    "success_rate": f"{high_conf_count / filtered_count * 100:.1f}%",
    "metadata_table": metadata_table
}

dbutils.notebook.exit(json.dumps(summary))

In [0]:
# Create sem_company_canonical - denormalized view of company name variations
# This table is expected by warehouse.dim_company and warehouse.dim_company_alias

print("\nCreating sem_company_canonical table...")

# Denormalize matched companies to show all name variations
canonical_df = high_confidence_df.select(
    F.col("canonical_company_name"),
    F.col("extracted_company_name").alias("company_name_raw"),
    F.col("normalized_name").alias("company_name_norm"),
    F.col("match_method").alias("company_match_method"),
    F.col("confidence").cast("decimal(5,4)").alias("company_match_confidence"),
    F.lit(True).alias("active_flag"),
    F.current_timestamp().alias("created_at"),
    F.current_timestamp().alias("updated_at")
)

# Write to semantic.sem_company_canonical
canonical_table = "workspace.semantic.sem_company_canonical"

print(f"Writing {canonical_df.count()} company variations to {canonical_table}...")

canonical_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(canonical_table)

print("✓ sem_company_canonical table created successfully")

# Show sample
print("\nSample from sem_company_canonical:")
display(spark.table(canonical_table).limit(10))